In [1]:
from datetime import date

import hisepy
import os
import tarfile
import session_info
import shutil

## Get the DEG app repo from Github

In [2]:
pipeline_repo = 'https://github.com/aifimmunology/sc-deg-explorer'
pipeline_sha = 'adec515c87a5b6ab183ddea9923b04bd0958efef'
pipeline_local_path = 'sc-deg-explorer'

In [7]:
if os.path.isdir(pipeline_local_path):
    shutil.rmtree(pipeline_local_path)

In [8]:
clone_command = f'git config --global url."https://github.com/".insteadOf "git@github.com:" ; git clone --recurse-submodules {pipeline_repo} {pipeline_local_path}; cd {pipeline_local_path}; git reset --hard {pipeline_sha}'

In [9]:
os.system(clone_command)

Cloning into 'sc-deg-explorer'...
Submodule 'libs/allen_dash_modules' (git@github.com:aifimmunology/allen-dash-modules.git) registered for path 'libs/allen_dash_modules'
Cloning into '/home/workspace/repro-vrd-tea-seq/data-apps/sc-deg-explorer/libs/allen_dash_modules'...


Submodule path 'libs/allen_dash_modules': checked out '5741af69b9efbef0823bf4a04a299d2cd240e392'
HEAD is now at adec515 README.md: uv --> pixi commands


0

## Get formatted files from HISE

In [12]:
results_uuid = 'b5f6de81-5ee3-4ed7-b737-e273065ec23c'
results_tar = hisepy.cache_files([results_uuid])[0]

2026-07-29 15:00:46,187 INFO [hisepy.logging:185] logging 18841 138178909071168 Calling cache_files
2026-07-29 15:00:51,553 INFO [hisepy.logging:228] logging 18841 138178909071168 Finished cache_files successfully (time_elapsed=2.953s)


In [13]:
with tarfile.open(results_tar) as tf:
    tf.extractall()

/tmp/ipykernel_18841/3870275805.py:2: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall()


## Copy files into the app directory

In [15]:
shutil.copytree('results', f'{pipeline_local_path}/results')

'sc-deg-explorer/results'

In [16]:
shutil.copyfile('deg-configs/tcell-vrd.json', f'{pipeline_local_path}/config/template.json')

'sc-deg-explorer/config/template.json'

## Deploy the app to HISE

In [18]:
current_ver = '0.1.0'

In [19]:
app_path = 'tcell-vrd-deg-app-{c}'.format(c = current_ver)
mount_path = app_path + '-mount'

In [20]:
app_files = ['app.py', 'pixi.lock', 'pixi.toml']
app_files = [f'{pipeline_local_path}/{f}' for f in app_files]
app_dirs = ['assets/', 'callbacks/', 'components/', 'config/', 'results/', 'libs/', 'ui/']
app_dirs = [f'{pipeline_local_path}/{f}' for f in app_dirs]

In [21]:
session_info.show()

In [25]:
result = hisepy.save_visualization_app(

    title=f'T cell VRd Treatment DEG App ({current_ver})',
    application_files = app_files,
    application_dirs = app_dirs,
    description = f'T cell VRd Treatment DEG App ({current_ver})',
    png_image = f'deg-configs/hero_image.png',

    data_mount_path = mount_path,
    data_source_file_ids = [results_uuid],

    study_space_id = '40df6403-29f0-4b45-ab7d-f46d420c422e',

    build_template_name= 'dash',
    build_template_parameters={
        'app_filepath': os.path.abspath(f'{pipeline_local_path}/app.py'),
        'requirements_filepath': os.path.abspath(f'{pipeline_local_path}/pixi.toml'),
    },
    build_template_major_version = 3,
    build_template_minor_version = 4,
    infer_build_template_arguments = False

)

2026-07-29 15:12:17,751 INFO [hisepy.logging:185] logging 18841 138178909071168 Calling save_visualization_app
2026-07-29 15:12:18,926 INFO [hisepy.logging:366] upload 18841 138178909071168 Created temporary directory for Visualization App build: /home/workspace/temp/tmp1wvsebgx
2026-07-29 15:12:20,994 INFO [hisepy.logging:185] logging 18841 138178909071168 Calling save_static_image
2026-07-29 15:12:27,567 INFO [hisepy.logging:228] logging 18841 138178909071168 Finished save_static_image successfully (time_elapsed=3.863s)
2026-07-29 15:12:27,568 INFO [hisepy.logging:385] upload 18841 138178909071168 Creating Visualization App workflow: https://allenimmunology.org/ide-nextgen/visualization/viz-app/workflow
2026-07-29 15:12:31,948 INFO [hisepy.logging:228] logging 18841 138178909071168 Finished save_visualization_app successfully (time_elapsed=11.858s)
